# Feature Engineering

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120})
BLUE, ORANGE, GREEN, RED = '#4C72B0', '#DD8452', '#55A868', '#C44E52'

PROC = Path('../data/processed')
RAW  = Path('../data/raw')

## User Features

In [ ]:
train    = pd.read_parquet(PROC / 'train.parquet')
ratings  = pd.read_parquet(PROC / 'ratings.parquet')
metadata = pd.read_parquet(RAW  / 'metadata_raw.parquet')

from src.data.features import build_user_features, build_item_features

user_feats = build_user_features(train, metadata)
print('User features shape:', user_feats.shape)
display(user_feats.head(5))
display(user_feats.describe())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

axes[0,0].hist(user_feats['mean_rating'],  bins=40, color=BLUE,   edgecolor='white')
axes[0,0].set_title('Mean Rating Given')
axes[0,0].set_xlabel('Mean Rating')

axes[0,1].hist(user_feats['rating_count'], bins=40, color=ORANGE, edgecolor='white', log=True)
axes[0,1].set_title('Rating Count (log scale)')
axes[0,1].set_xlabel('# Ratings')

axes[1,0].hist(user_feats['rating_std'].clip(upper=2), bins=40, color=GREEN, edgecolor='white')
axes[1,0].set_title('Rating Std')
axes[1,0].set_xlabel('Std of Ratings')

top_genres = user_feats['top_genre'].value_counts().head(12)
axes[1,1].barh(top_genres.index[::-1], top_genres.values[::-1], color=BLUE)
axes[1,1].set_title('Primary Genre per User')
axes[1,1].set_xlabel('# Users')

plt.suptitle('User Feature Distributions', fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('figures/07_user_features.png', bbox_inches='tight')
plt.show()

## Item Features

In [ ]:
item_feats = build_item_features(ratings, metadata)
print('Item features shape:', item_feats.shape)
display(item_feats.head(5))
display(item_feats.describe())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

axes[0,0].hist(item_feats['mean_rating'], bins=40, color=BLUE, edgecolor='white')
axes[0,0].set_title('Item Mean Rating')

axes[0,1].hist(item_feats['rating_count'], bins=50, color=ORANGE, edgecolor='white', log=True)
axes[0,1].set_title('Rating Count (log)')

axes[0,2].hist(item_feats['description_len'].clip(upper=3000), bins=50, color=GREEN, edgecolor='white')
axes[0,2].set_title('Description Length (chars)')
axes[0,2].axvline(50, color=RED, linestyle='--', label='50-char threshold')
axes[0,2].legend()

desc_counts = item_feats['has_description'].value_counts()
axes[1,0].bar(['Has Description', 'Missing'], [desc_counts.get(True, 0), desc_counts.get(False, 0)],
               color=[GREEN, RED])
axes[1,0].set_title('Description Coverage')
axes[1,0].set_ylabel('# Books')

ptier = item_feats['price_tier'].value_counts()
axes[1,1].bar(ptier.index, ptier.values, color=ORANGE)
axes[1,1].set_title('Price Tier Distribution')

top_item_genres = item_feats['genre'].value_counts().head(10)
axes[1,2].barh(top_item_genres.index[::-1], top_item_genres.values[::-1], color=BLUE)
axes[1,2].set_title('Top 10 Genres')

plt.suptitle('Item Feature Distributions', fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('figures/08_item_features.png', bbox_inches='tight')
plt.show()

## Feature Correlation

In [ ]:
numeric_user = user_feats[['mean_rating', 'rating_count', 'rating_std', 'days_active']]
numeric_item = item_feats[['mean_rating', 'rating_count', 'rating_std', 'description_len']]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.heatmap(numeric_user.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=axes[0], square=True)
axes[0].set_title('User Feature Correlation')

sns.heatmap(numeric_item.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=axes[1], square=True)
axes[1].set_title('Item Feature Correlation')

plt.suptitle('Feature Correlation Matrices', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('figures/09_correlations.png', bbox_inches='tight')
plt.show()

## Feature Inventory

| Feature | Used | Notes |
|---|---|---|
| `mean_rating` (user) | ✗ | Side feature for LightFM / two-tower extensions |
| `rating_std` (user) | ✗ | Candidate sample weight in future training |
| `top_genre` (user) | ✗ | Side feature for content-augmented models |
| `days_active` (user) | ✗ | User maturity proxy |
| `rating_count` (item) | ✗ | Implicit in BPR embeddings |
| `has_description` (item) | ✓ | Gates semantic search indexing |
| `genre` (item) | ✓ | Injected into embedding text field |
| `price_tier` (item) | ✗ | Business-constrained ranking extension |